# Multi-Fock Interference on the Blue Sideband

**Purpose.** Predict the coherent sideband response of a displaced state $|\alpha\rangle$ with $\alpha = 4$ outside the Lamb–Dicke regime ($\eta \in [0, 3]$), taking the **signs** of the Laguerre polynomials seriously.

**Three steps, as discussed:**
1. Heatmap of $\Omega_{n,n+1}(\eta)$ for $n \in [8, 24]$, $\eta \in [0, 3]$.
2. Overlay with Glauber weights $|c_n|^2$ for $\alpha = 4$.
3. Coherent sum — with signs — as prediction for the carpet.

Conventions: Wineland et al. 1998 / Leibfried et al. 2003.
$$\Omega_{n,n+1}(\eta) = \Omega_0 \, e^{-\eta^2/2} \, \eta \, \frac{L_n^{(1)}(\eta^2)}{\sqrt{n+1}}.$$
$\Omega_0 = 1$ (arbitrary units). The $L_n^{(1)}$ sign is **kept** throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import eval_genlaguerre, factorial

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Core functions

In [ ]:
def rabi_bsb(n, eta):
    """Blue sideband Rabi frequency Omega_{n, n+1}(eta), Omega_0 = 1.
    
    Sign of L_n^{(1)}(eta^2) is preserved. n and eta broadcast.
    """
    n = np.asarray(n)
    eta = np.asarray(eta)
    L = eval_genlaguerre(n, 1, eta**2)
    return np.exp(-eta**2 / 2) * eta * L / np.sqrt(n + 1)

def coherent_amplitudes(alpha, n_max):
    """Glauber coherent state amplitudes c_n = e^{-|alpha|^2/2} alpha^n / sqrt(n!).
    
    Returns complex c_n for n = 0..n_max. For real alpha, c_n are real and positive.
    """
    n = np.arange(n_max + 1)
    # log form to stay numerically safe for alpha = 4, n ~ 30
    log_cn = -0.5 * np.abs(alpha)**2 + n * np.log(alpha + 0j) - 0.5 * np.log(factorial(n).astype(float))
    return np.exp(log_cn)

## Step 1 — Heatmap of $\Omega_{n,n+1}(\eta)$ with signs

Diverging colormap so sign changes of $L_n^{(1)}$ are visible. Zero crossings are where adjacent Fock components flip their contribution to the coherent sum.

In [ ]:
n_vals = np.arange(8, 25)          # n in [8, 24]
eta_vals = np.linspace(0, 3, 400)

N, E = np.meshgrid(n_vals, eta_vals, indexing='ij')
Omega = rabi_bsb(N, E)

vmax = np.abs(Omega).max()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(eta_vals, n_vals, Omega, cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, shading='auto')
ax.set_xlabel(r'Lamb–Dicke parameter $\eta$')
ax.set_ylabel(r'Fock number $n$')
ax.set_title(r'Blue sideband Rabi frequency $\Omega_{n,n+1}(\eta)/\Omega_0$ — signs preserved')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'$\Omega_{n,n+1}/\Omega_0$')

# overlay contour at zero to make sign flips explicit
ax.contour(eta_vals, n_vals, Omega, levels=[0], colors='k', linewidths=0.8, linestyles='--')
plt.tight_layout()
plt.show()

**Read this as:** dashed contours mark $\Omega_{n,n+1}=0$. Crossing a contour in either direction flips the sign of that Fock component's contribution to the coherent sideband response. In the Lamb–Dicke limit ($\eta \to 0$) there are no such flips; for $\eta \gtrsim 1$ with $n \sim \bar n = 16$, several flips occur across the populated band.

## Step 2 — Overlay with Glauber weights $|c_n|^2$ for $\alpha = 4$

In [ ]:
alpha = 4.0
n_max = 40
c_n = coherent_amplitudes(alpha, n_max)
p_n = np.abs(c_n)**2

n_full = np.arange(n_max + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={'width_ratios': [1, 2]})

# left: Glauber distribution
ax1.bar(n_full, p_n, color='#2c5f7c', alpha=0.85)
ax1.axvspan(8, 24, color='#c0392b', alpha=0.12, label='window [8, 24]')
ax1.set_xlabel(r'$n$')
ax1.set_ylabel(r'$|c_n|^2$')
ax1.set_title(rf'Glauber weights, $\alpha={alpha}$, $\bar n = {alpha**2:.0f}$')
ax1.legend(loc='upper right', fontsize=8)

# right: weighted sideband amplitude |c_n|^2 * Omega_{n,n+1}(eta) — signs kept
eta_line = np.linspace(0, 3, 600)
for n in n_vals:
    w = p_n[n]
    ax2.plot(eta_line, w * rabi_bsb(n, eta_line), alpha=0.7, linewidth=1.2,
             label=f'n={n}' if n in (10, 14, 16, 18, 22) else None)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xlabel(r'$\eta$')
ax2.set_ylabel(r'$|c_n|^2 \, \Omega_{n,n+1}(\eta)/\Omega_0$')
ax2.set_title('Weighted contributions (signs kept)')
ax2.legend(ncol=5, fontsize=8, loc='lower left')
plt.tight_layout()
plt.show()

# Sanity: window coverage
window_mass = p_n[8:25].sum()
print(f'Probability mass in window n ∈ [8, 24]: {window_mass:.3f}')
print(f'Mass outside (n<8 or n>24): {1 - window_mass:.3f}')

If the window misses more than ~10% mass, widen it in the next cell (set `n_window`). We keep the plot window at [8, 24] for readability but compute the coherent sum over the **full** populated band.

## Step 3 — Coherent sum as carpet prediction

**Two observables.** Let $\delta = \omega_L - (\omega_0 + \omega_z)$ be the detuning from the blue sideband. For a coherent state, the sideband excitation after a pulse of duration $t$ is

$$P_\uparrow(\delta, t) = \sum_n |c_n|^2 \, \frac{\Omega_{n,n+1}^2}{\Omega_{n,n+1}^2 + \delta^2} \, \sin^2\!\left(\tfrac{1}{2}\sqrt{\Omega_{n,n+1}^2 + \delta^2}\, t\right).$$

This is an **incoherent** sum over $n$ because different $n$ drive different final Fock states $|n{+}1\rangle$ — they are orthogonal motional outcomes.

The **coherent** (sign-sensitive) signature appears in a **Ramsey / echo / stroboscopic** sequence where the motional coherence between $|n\rangle$ and a reference is preserved. The simplest proxy that exposes the Laguerre sign structure is the *amplitude* sum

$$\mathcal{A}(\eta) = \sum_n |c_n|^2 \, \frac{\Omega_{n,n+1}(\eta)}{\Omega_0},$$

which is the leading-order Ramsey contrast envelope under a short $\pi/2$ pulse. We plot **both**: the incoherent carpet and the coherent envelope — the contrast between them is the prediction.

In [ ]:
# -------- incoherent carpet P_up(delta, t) at fixed eta --------
eta_fixed = 2.0   # choose experimentally relevant value; adjust as needed
t_vals = np.linspace(0, 8 * np.pi, 300)       # in units of 1/Omega_0
delta_vals = np.linspace(-3, 3, 300)          # in units of Omega_0

# truncate to n where p_n is non-negligible
n_active = np.where(p_n > 1e-5)[0]

Omega_n = rabi_bsb(n_active, eta_fixed)       # signed; we square below so sign drops here
w_n = p_n[n_active]

T, D = np.meshgrid(t_vals, delta_vals, indexing='ij')
P = np.zeros_like(T)
for k, n in enumerate(n_active):
    On2 = Omega_n[k]**2
    Oeff = np.sqrt(On2 + D**2)
    P += w_n[k] * On2 / (On2 + D**2) * np.sin(0.5 * Oeff * T)**2

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(delta_vals, t_vals, P, cmap='magma', shading='auto')
ax.set_xlabel(r'detuning $\delta/\Omega_0$')
ax.set_ylabel(r'pulse time $\Omega_0 t$')
ax.set_title(rf'Predicted carpet $P_\uparrow(\delta, t)$ at $\eta = {eta_fixed}$, $\alpha = {alpha}$')
fig.colorbar(im, ax=ax, label=r'$P_\uparrow$')
plt.tight_layout()
plt.show()

In [ ]:
# -------- coherent envelope A(eta) — sign-sensitive --------
eta_scan = np.linspace(0, 3, 600)

A_coherent = np.zeros_like(eta_scan)
A_incoherent_abs = np.zeros_like(eta_scan)   # sum of |p_n * Omega| — what you'd get ignoring signs

for n in n_active:
    Om = rabi_bsb(n, eta_scan)
    A_coherent        += p_n[n] * Om         # signs preserved — interference happens here
    A_incoherent_abs  += p_n[n] * np.abs(Om) # what a sign-blind analysis would predict

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(eta_scan, A_coherent, color='#c0392b', lw=2,
        label=r'coherent: $\sum_n |c_n|^2 \, \Omega_{n,n+1}(\eta)$ (signs kept)')
ax.plot(eta_scan, A_incoherent_abs, color='#2c5f7c', lw=1.5, ls='--',
        label=r'sign-blind: $\sum_n |c_n|^2 \, |\Omega_{n,n+1}(\eta)|$')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(r'$\eta$')
ax.set_ylabel(r'amplitude / $\Omega_0$')
ax.set_title(r'Coherent vs sign-blind envelope for $\alpha = 4$')
ax.legend()
plt.tight_layout()
plt.show()

# Locate sign changes of the coherent envelope — these are the predicted contrast nodes
sign_changes = np.where(np.diff(np.sign(A_coherent)))[0]
print('Predicted nodes of coherent envelope at eta ≈',
      [f'{eta_scan[i]:.3f}' for i in sign_changes])

## What to compare against the data

1. **Carpet plot above** — compare shape of $P_\uparrow(\delta, t)$ to Freddy's measured carpet at the same $\eta$. Periodic contrast revivals along $t$ at fixed $\delta$ should appear naturally from the incoherent sum if the Fock distribution and $\Omega_n$ are right.
2. **Coherent envelope** — the red curve predicts $\eta$-values where the Ramsey/echo contrast **vanishes by destructive interference of adjacent Fock components**. These nodes are the distinctive signature; a sign-blind model (blue dashed) cannot produce them.
3. **Validation test.** If the experimental $\eta$ can be scanned (or inferred from beam geometry), the predicted node positions are a falsifier. If contrast drops coincide with red-curve zero crossings, multi-Fock interference is confirmed. If not, we look elsewhere.

**Parameters to vary before the meeting:**
- `alpha` — the displacement; changes Fock envelope and thus which $L_n^{(1)}$ zeros lie inside the populated band.
- `eta_fixed` in the carpet cell — match the experimental $\eta$.
- Pulse duration range `t_vals` — match the Ramsey / stroboscopic sequence actually used.

## Step 4 — Pulse-train Fourier spectrum

Steps 1–3 used the single-sideband matrix element $\Omega_{n,n+1}(\eta)$. But a *stroboscopic pulse train* at period $T_m = 2\pi/\omega_m$ is a multi-frequency drive: its Fourier series is

$$\text{mod}(t) = d + \sum_{k\neq 0} F_k \, e^{i k \omega_m t}, \qquad F_k = \frac{\sin(\pi k d)}{\pi k},$$

with $d = \delta t / T_m$ the duty cycle. Each harmonic $k \in \mathbb{Z}$ is resonant with a different sideband: $k=0$ carrier, $k=\pm 1$ first sideband, $k=\pm 2$ second, etc. In the non-LD regime each sideband has its *own* Laguerre structure; the pulse train opens *all* these channels simultaneously with weights $F_k$.

In [ ]:
def pulse_train_fourier(k, d):
    """Fourier coefficient F_k of a unit-amplitude stroboscopic square wave.

    mod(t) = d  +  Σ_{k ≠ 0}  F_k · exp(i k ω_m t),   F_k = sin(π k d)/(π k).
    """
    k = np.asarray(k)
    return np.where(k == 0, d, np.sin(np.pi * k * d) / (np.pi * k + (k == 0)))

k_range = np.arange(-6, 7)
duty_cycles = [0.05, 0.13, 0.30, 0.50]   # fast-AOM, slow-AOM, wide, half

fig, ax = plt.subplots(figsize=(8, 4))
for d in duty_cycles:
    F = pulse_train_fourier(k_range, d)
    ax.stem(k_range + 0.08*duty_cycles.index(d), np.abs(F),
            basefmt=' ',
            label=fr'$d = {d:.2f}$', linefmt=f'C{duty_cycles.index(d)}-',
            markerfmt=f'C{duty_cycles.index(d)}o')
ax.set_xlabel(r'harmonic $k$   (resonant detuning $\delta = k\,\omega_m$)')
ax.set_ylabel(r'$|F_k|$')
ax.set_title('Stroboscopic pulse-train Fourier amplitudes')
ax.set_xticks(k_range)
ax.axhline(0, color='k', lw=0.4)
ax.legend(ncol=4, fontsize=9, loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'fast-AOM duty (δt=40 ns, T_m=770 ns): d ≈ 0.052 → |F_1|/|F_0| ≈',
      f'{np.abs(pulse_train_fourier(1, 0.052))/0.052:.3f}')
print(f'slow-AOM duty (δt=100 ns):           d ≈ 0.130 → |F_1|/|F_0| ≈',
      f'{np.abs(pulse_train_fourier(1, 0.130))/0.130:.3f}')

**Reading the spectrum.** At d = 0.05 (fast AOM, ~40 ns pulses) the sideband harmonics $|F_{\pm 1}|, |F_{\pm 2}|$ are nearly equal and within a factor ~3 of the carrier coefficient $d$. So the pulse train is *far from* a clean carrier drive: the first few sidebands all receive comparable Fourier amplitude. At larger duty cycle $d = 0.5$ the spectrum collapses toward a pure carrier. The ion sees all these channels simultaneously.

## Step 5 — Open channels per sideband: $A_k(\eta)$

For each harmonic $k$, the Ramsey-contrast amplitude on that sideband for a coherent state $|\alpha\rangle$ is

$$A_k(\eta) = \sum_n c_n^* \, c_{n+|k|} \, \Omega_{n, n+|k|}(\eta),$$

with the generalised sideband matrix element (Wineland et al. 1998, Leibfried 2003):

$$\Omega_{n, n+|k|}(\eta) = \Omega_0 \, e^{-\eta^2/2} \, \eta^{|k|} \sqrt{\frac{n!}{(n+|k|)!}} \, L_n^{(|k|)}(\eta^2).$$

For $k=0$ this reduces to the carrier matrix element; for $k=1$ it reproduces the rabi_bsb function of Step 1. The *sign* of $L_n^{(|k|)}$ is kept. Below we plot $A_k(\eta)$ for $k \in \{-3, ..., +3\}$ at fixed $\alpha = 4$.

In [ ]:
def rabi_sideband(n, eta, k):
    """Ω_{n, n+|k|}(η) / Ω_0, Laguerre-signed. k=0 → carrier; k≠0 → |k|-th sideband."""
    n = np.asarray(n); eta = np.asarray(eta)
    k_abs = abs(int(k))
    L = eval_genlaguerre(n, k_abs, eta**2)
    if k_abs == 0:
        return np.exp(-eta**2 / 2) * L
    from scipy.special import gammaln
    log_ratio = 0.5 * (gammaln(n + 1) - gammaln(n + k_abs + 1))
    return np.exp(-eta**2 / 2 + log_ratio) * eta**k_abs * L


def channel_amplitude(alpha, eta, k, n_max=60):
    """A_k(η) = Σ_n c_n^* c_{n+|k|} · Ω_{n, n+|k|}(η),   signed."""
    c = coherent_amplitudes(alpha, n_max)
    eta = np.asarray(eta)
    k_abs = abs(int(k))
    if k_abs == 0:
        # Carrier: Σ_n |c_n|² · Ω_{n,n}(η)
        n_arr = np.arange(n_max + 1)
        scalar = (eta.ndim == 0)
        eta_b = eta.reshape(-1) if not scalar else eta[None]
        out = np.array([np.sum(np.abs(c)**2 * rabi_sideband(n_arr, e, 0)) for e in eta_b])
        return out.item() if scalar else out.reshape(eta.shape)
    # Sideband: coherent sum over shifted pair
    n_arr = np.arange(n_max + 1 - k_abs)
    scalar = (eta.ndim == 0)
    eta_b = eta.reshape(-1) if not scalar else eta[None]
    out = np.zeros(eta_b.shape, dtype=complex)
    for j, e in enumerate(eta_b):
        Om = rabi_sideband(n_arr, e, k_abs)
        out[j] = np.sum(np.conj(c[:n_max+1-k_abs]) * c[k_abs:n_max+1] * Om)
    if k < 0:
        out = np.conj(out)
    return out.item() if scalar else out.reshape(eta.shape)


alpha_fixed = 4.0
eta_scan = np.linspace(0.05, 3.0, 400)
k_list = [-3, -2, -1, 0, 1, 2, 3]

A_k = {k: channel_amplitude(alpha_fixed, eta_scan, k, n_max=60) for k in k_list}

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(k_list)))
for k, col in zip(k_list, colors):
    y = A_k[k].real if np.iscomplexobj(A_k[k]) else A_k[k]
    ax.plot(eta_scan, y, lw=1.5, color=col, label=fr'$k = {k:+d}$')
ax.axhline(0, color='k', lw=0.4)
ax.set_xlabel(r'$\eta$')
ax.set_ylabel(r'$\mathrm{Re}\,A_k(\eta) / \Omega_0$')
ax.set_title(rf'Open channel amplitudes per sideband ($\alpha = {alpha_fixed}$)')
ax.legend(ncol=7, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Sign-change counts (nodes of each channel over η ∈ [0.05, 3.0]):')
for k in k_list:
    y = A_k[k].real if np.iscomplexobj(A_k[k]) else A_k[k]
    nodes = int(np.sum(np.diff(np.sign(y)) != 0))
    print(f'  k = {k:+d}:  {nodes} sign changes')

**Reading Step 5.** $k = 0$ and $k = \pm 1$ each have their own Laguerre-sign structure — the $k=+1$ curve reproduces the notebook's original $A(\eta)$ from Step 3 (the single-channel envelope). $k = \pm 2, \pm 3$ are suppressed by $\eta^2, \eta^3$ at small $\eta$ but catch up in the non-LD regime ($\eta \gtrsim 1$), and their node positions are *different* from the $k=1$ nodes. A physical measurement probes the Fourier-amplitude-weighted sum of all these — which is what Step 6 computes.

## Step 6 — Weighted multi-channel total

The full stroboscopic pulse train couples to the spin through *every* open channel with weight $F_k$ from Step 4. To leading order in pulse area, the multi-channel Ramsey envelope is

$$A_\text{train}(\eta) = \sum_{k} F_k(d) \, A_k(\eta),$$

with signs kept throughout. Below we overlay $A_\text{train}(\eta)$ against Step 3's single-channel $A(\eta) = A_{k=+1}(\eta)$. The difference exposes the contribution of the carrier and higher sidebands that the single-sideband picture misses.

In [ ]:
# compute A_train(η) at three representative duty cycles
duty_picks = [0.05, 0.13, 0.30]
k_range_full = np.arange(-4, 5)   # carrier ± 4 sidebands — enough for η ≤ 3 at α=4

# reuse A_k computed above for k in [-3, 3]; compute ±4 on the fly
A_k_full = dict(A_k)
for k in (-4, 4):
    A_k_full[k] = channel_amplitude(alpha_fixed, eta_scan, k, n_max=60)

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True, constrained_layout=True)

ax = axs[0]
# Step-3 single-channel reference
A_single = A_k_full[+1].real
ax.plot(eta_scan, A_single, color='black', lw=1.8, ls='--',
        label=r'Step 3 single channel: $A_{+1}(\eta)$')
# Multi-channel at three duty cycles
for d in duty_picks:
    A_train = np.zeros_like(eta_scan)
    for k in k_range_full:
        F = pulse_train_fourier(k, d)
        y = A_k_full[k].real if np.iscomplexobj(A_k_full[k]) else A_k_full[k]
        A_train += F * y
    ax.plot(eta_scan, A_train, lw=1.5, label=fr'multi-channel, $d = {d}$')
ax.axhline(0, color='k', lw=0.4)
ax.set_ylabel(r'amplitude / $\Omega_0$')
ax.set_title(rf'Multi-channel Ramsey envelope $A_\mathrm{{train}}(\eta)$ vs single channel ($\alpha = {alpha_fixed}$)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Bottom panel: residual (multi - single) at the fast-AOM duty
d_fast = 0.05
A_train_fast = np.zeros_like(eta_scan)
for k in k_range_full:
    F = pulse_train_fourier(k, d_fast)
    y = A_k_full[k].real if np.iscomplexobj(A_k_full[k]) else A_k_full[k]
    A_train_fast += F * y
residual = A_train_fast - A_single
ax = axs[1]
ax.plot(eta_scan, residual, color='#c0392b', lw=1.5,
        label=fr'residual at $d = {d_fast}$')
ax.axhline(0, color='k', lw=0.4)
# Mark original Step-3 nodes
from numpy import diff, sign
step3_nodes = eta_scan[np.where(diff(sign(A_single)))[0]]
for e in step3_nodes:
    ax.axvline(e, color='grey', lw=0.4, ls=':', alpha=0.7)
ax.set_xlabel(r'$\eta$')
ax.set_ylabel(r'$A_\mathrm{train} - A_{+1}$')
ax.set_title('Residual — extra structure added by carrier + higher sidebands')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

# Node positions at d = 0.05
train_nodes = eta_scan[np.where(diff(sign(A_train_fast)))[0]]
print(f'Step-3 single-channel nodes:    {[f"{e:.3f}" for e in step3_nodes]}')
print(f'Multi-channel nodes (d=0.05):   {[f"{e:.3f}" for e in train_nodes]}')

## What this extension adds

1. **Step 4** makes explicit that a stroboscopic pulse train is a multi-tone drive: with 5% duty cycle (fast-AOM regime) the $k = \pm 1, \pm 2$ Fourier coefficients are all within a factor ~3 of the carrier. So single-sideband analysis is already a strong assumption even for a "BSB π/2" pulse.

2. **Step 5** extends the Laguerre/Glauber machinery of Steps 1–3 to *every* open channel. Each channel has its own sign structure, its own node locations, and its own small-η scaling ($\propto \eta^{|k|}$).

3. **Step 6** predicts the *actual* measurable envelope: a $F_k$-weighted coherent sum over channels. Node positions shift relative to the single-channel prediction, and the residual (bottom panel of Step 6) is exactly the signature of "other channels are active too". A falsification test: if Freddy's contrast nodes line up with Step-3 nodes → single-channel picture survives; if they line up with Step-6 multi-channel nodes → evidence for coherent multi-channel physics.

**Connection to the engine (WP-V Round 2+):** our `stroboscopic_sweep.py` engine's full unitary $\exp(-i H \delta t)$ with $H = \omega_m a^\dagger a + (\Omega/2)(C \sigma_- + C^\dagger \sigma_+)$ automatically includes every $A_k$. A direct cross-check against this notebook at selected $(\alpha, \eta, d)$ verifies the engine's multi-sideband content against the Laguerre closed form.